# Debug List Scraper Seputar Malang
List-only. Halaman list saat ini tidak menyediakan published_date, jadi tanggal akan `None` kecuali selector list berubah.

In [1]:
print("s")

s


In [2]:
from pathlib import Path
import sys
import importlib
import inspect

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, fetch_text, parse_date, normalize_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.seputarmalang as scraper
scraper = importlib.reload(scraper)
print('module file:', scraper.__file__)


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/seputarmalang.py


In [3]:
MAX_PAGES = 200
MAX_CLICKS = 40
TITLE_LIMIT = 90


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(
            rows=('url', 'count'),
            newest=('published_dt', 'max'),
            oldest=('published_dt', 'min'),
        )
        .reset_index()
        .tail(60)
    )
    return df


In [4]:
def debug_scrape_list():
    rows = []
    url = scraper.BASE_URL
    print('Scraping Seputar Malang page 1')
    soup = fetch_html(url)
    for card in soup.select('article.jeg_post'):
        title_tag = card.select_one('h3.jeg_post_title a')
        date_tag = card.select_one('.jeg_meta_date a, .jeg_meta_date')
        if not title_tag:
            continue
        rows.append({
            'page_num': 1,
            'list_page_url': url,
            'title': title_tag.get_text(strip=True),
            'url': title_tag['href'],
            'published_date': date_tag.get_text(' ', strip=True) if date_tag else None,
        })
    return records_to_df(rows)

list_df = debug_scrape_list()
list_df = ensure_debug_columns(list_df)
print_debug_rows(list_df)


Scraping Seputar Malang page 1
page=001 | date=14 Februari 2026 | title=Gebyar Penutupan KKN-T 2026 Unira Malang Gelar Expo Loka Wisata
page=001 | date=26 Januari 2026 | title=Peringati Isra Mikraj di Desa Selorejo, KKNT 14 UNIRA Malang Hadirkan Alumni AKSI Indosiar
page=001 | date=23 Desember 2025 | title=Mahasiswa KKN Desa Wisata 2026 Unira Malang Antusias Mengikuti Pelatihan Jurnalistik da...
page=001 | date=17 Desember 2025 | title=Gus Syafiq, Kasatkornas Banser Tekankan Kader untuk Aktif Jaga Kedaulatan Pangan Nasional
page=001 | date=11 September 2025 | title=KKN STAI An-Najah Indonesia Mandiri Gelar Lomba Puisi dan Pidato di Sidodadi
page=001 | date=27 Agustus 2025 | title=Unira Malang Menjadi Ruang Apresiasi Kelompok Tani Kopi, Hadirkan Coffee Traveler dari ...
page=001 | date=21 Agustus 2025 | title=GP Ansor Jawa Timur Gandeng Unira Malang Bagikan 150 Ton Benih Padi untuk Ketahanan Pangan
page=001 | date=17 Agustus 2025 | title=Peringatan HUT RI ke-80 di Gedangan, 70 Siswa SMK

In [5]:
list_df = audit_list(list_df)


total rows: 15
first page: 1
last page: 1
newest date: 2026-06-26 00:00:00
oldest date: 2025-06-11 00:00:00
cutoff date: 2026-02-28 08:50:47.251595
reached cutoff: True
null parsed dates: 0

rows per month:


,month,count
0,2025-06,1
1,2025-08,4
2,2025-09,1
3,2025-12,2
4,2026-01,1
5,2026-02,1
6,2026-05,3
7,2026-06,2



rows per page:


,page_num,rows,newest,oldest
0,1,15,2026-06-26,2025-06-11


In [6]:
output_path = PROJECT_ROOT / 'csv' / 'seputarmalang_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/seputarmalang_list_debug.csv')

## Optional article sample check

Ambil beberapa artikel detail, cek `content`, lalu simpan ke article debug CSV.

In [ ]:

ARTICLE_SAMPLE_SIZE = 5
ARTICLE_DEBUG_OUTPUT_PATH = PROJECT_ROOT / 'csv' / 'seputar_malang_article_debug.csv'

article_rows = []
sample_rows = list_df.head(ARTICLE_SAMPLE_SIZE)
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")

article_df = pd.DataFrame(article_rows)
ARTICLE_DEBUG_OUTPUT_PATH.parent.mkdir(exist_ok=True)
article_df.to_csv(ARTICLE_DEBUG_OUTPUT_PATH, index=False)
print('saved:', ARTICLE_DEBUG_OUTPUT_PATH)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
